# Project 4: Listening Past the Noise — Submission 10
### Flagship Classical ML Architecture: Full-Data Retraining + TPP + Dual Kernels (VLAD + PCA Fisher)

**Score & Key Highlights:**
- **Kaggle Public Leaderboard:** **0.84353 MAP@3**
- **Full-Data Retraining:** All final classifiers are fitted on **100% of the 620 verified recordings**, overcoming fold subsampling for rare sound categories (10 samples).
- **Multi-Instance Temporal Pyramid Pooling (TPP, 768 dims):** Captures temporal sound progression (Start, Mid, End) across dual log-mel frame widths without neural networks.
- **Dual-Kernel Stacking:** Compounding both the **VLAD residual Gram matrix** and the **PCA-whitened GMM Fisher Vector kernel matrix** via Linear Support Vector Machines.
- **Semi-Supervised Agreement Filtering:** Harvests high-confidence pseudo-labels from the 4,984 unchecked audio recordings.
- **Exact Calibration:** Temperature sharpening ($T=1.25$) and test-set Expectation-Maximization prior matching ($\pi^{0.5}$).

In [ ]:
import os, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import soundfile as sf, librosa
warnings.filterwarnings("ignore")

# Primary dataset path: Kaggle competition input directory
DATA_DIR = Path("/kaggle/input/competitions/project4")
if not DATA_DIR.exists():
    LOCAL_FALLBACK = Path("/home/pranav/WORK/Work/IIITD/ML/Kaggle_comp/project4")
    if LOCAL_FALLBACK.exists():
        DATA_DIR = LOCAL_FALLBACK
    else:
        matches = list(Path("/kaggle/input").rglob("train.csv"))
        DATA_DIR = matches[0].parent if matches else Path("project4")

# Working directory for Kaggle output / caching
WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path(".")

# Load dataset metadata
train_df  = pd.read_csv(DATA_DIR / "train.csv", dtype={"id": str})
test_df   = pd.read_csv(DATA_DIR / "test.csv",  dtype={"id": str})
sample_df = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})

# Audio directory paths
if (DATA_DIR / "audio/audio/train").exists():
    TRAIN_AUDIO = DATA_DIR / "audio/audio/train"
    TEST_AUDIO  = DATA_DIR / "audio/audio/test"
elif (DATA_DIR / "audio/train").exists():
    TRAIN_AUDIO = DATA_DIR / "audio/train"
    TEST_AUDIO  = DATA_DIR / "audio/test"
else:
    train_audio_matches = [p for p in DATA_DIR.rglob("train") if p.is_dir()]
    test_audio_matches  = [p for p in DATA_DIR.rglob("test") if p.is_dir()]
    TRAIN_AUDIO = train_audio_matches[0] if train_audio_matches else DATA_DIR / "audio/train"
    TEST_AUDIO  = test_audio_matches[0] if test_audio_matches else DATA_DIR / "audio/test"

print(f"Data directory: {DATA_DIR}")
print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")
print(f"Train audio: {TRAIN_AUDIO} | Test audio: {TEST_AUDIO}")

from joblib import Parallel, delayed, dump, load
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import MiniBatchKMeans
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics.pairwise import linear_kernel

classes = np.array(sorted(train_df["label"].unique()))
K       = len(classes)
cls_idx = {c: i for i, c in enumerate(classes)}

verified_mask = (train_df["checked"].values == 1)
y_all_idx     = train_df["label"].map(cls_idx).values
yv            = y_all_idx[verified_mask]
yu            = y_all_idx[~verified_mask]

verified_ids   = np.flatnonzero(verified_mask)
unchecked_ids  = np.flatnonzero(~verified_mask)
test_ids       = np.arange(len(train_df), len(train_df) + len(test_df))

print(f"Categories: {K} | Verified: {len(yv)} | Unchecked: {len(yu)} | Test: {len(test_df)}")

## 1. Feature Extraction: 319-Dim Rich Features & Multi-Scale Patches

In [ ]:
SR, N_FFT, HOP = 22050, 2048, 512

def extract_one_rich(path):
    try:
        y, sr = sf.read(str(path), dtype="float32")
        if y.ndim > 1: y = y.mean(axis=1)
        dur = len(y) / sr
        if len(y) < N_FFT: y = np.pad(y, (0, N_FFT - len(y)))
        S = np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP)); P2 = S ** 2
        rms  = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP)
        zcr  = librosa.feature.zero_crossing_rate(y=y, frame_length=N_FFT, hop_length=HOP)
        mel128 = librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=128)
        mfcc   = librosa.feature.mfcc(S=librosa.power_to_db(mel128), n_mfcc=20)
        chroma = librosa.feature.chroma_stft(S=P2, sr=sr)
        cent   = librosa.feature.spectral_centroid(S=S, sr=sr)
        bw     = librosa.feature.spectral_bandwidth(S=S, sr=sr)
        roll   = librosa.feature.spectral_rolloff(S=S, sr=sr)
        con    = librosa.feature.spectral_contrast(S=S, sr=sr)
        ms = lambda M: np.concatenate([M.mean(axis=1), M.std(axis=1)])
        base = np.concatenate([ms(mfcc), ms(chroma), ms(cent), ms(bw), ms(roll), ms(zcr), ms(rms), [dur]])
        contrast = ms(con)

        r1 = rms[0]; T = S.shape[1]
        act = r1 >= max(np.percentile(r1, 30), 0.02 * r1.max())
        if act.sum() < 3: act = np.ones(T, bool)
        lm  = librosa.power_to_db(librosa.feature.melspectrogram(S=P2, sr=sr, n_mels=40))
        lma, mfa = lm[:, act], mfcc[:, act]
        dm  = librosa.feature.delta(mfcc, width=9 if T >= 9 else 3, mode="nearest")
        flat = librosa.feature.spectral_flatness(S=S)[0]
        env  = librosa.onset.onset_strength(S=lm, sr=sr, hop_length=HOP)
        n_on = len(librosa.onset.onset_detect(onset_envelope=env, sr=sr, hop_length=HOP))
        lr   = np.log(r1 + 1e-8)
        rich = np.concatenate([
            lma.mean(1), lma.std(1), np.percentile(lma, 90, axis=1), lm.max(1),
            mfa.mean(1), mfa.std(1), dm.std(1),
            [flat.mean(), flat.std(), env.mean(), env.std()],
            np.percentile(lr, [10, 50, 90]),
            [act.mean(), np.log(dur + 1e-3), n_on / max(dur, 1e-3)]])
        return np.concatenate([base, contrast, rich]).astype(np.float32)
    except Exception as e:
        return None

FEAT_CACHE = WORK / "p4_features_rich.npz"
if FEAT_CACHE.exists():
    z = np.load(FEAT_CACHE, allow_pickle=True)
    Xtr_all, Xte_all = z["Xtr"], z["Xte"]
else:
    train_rich = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_one_rich)(TRAIN_AUDIO / f"{fid}.wav") for fid in train_df["id"]
    )
    test_rich = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_one_rich)(TEST_AUDIO / f"{fid}.wav") for fid in test_df["id"]
    )
    Xtr_all = np.vstack(train_rich); Xte_all = np.vstack(test_rich)
    med = np.nanmedian(Xtr_all, axis=0)
    Xtr_all = np.where(np.isfinite(Xtr_all), Xtr_all, med)
    Xte_all = np.where(np.isfinite(Xte_all), Xte_all, med)
    np.savez_compressed(FEAT_CACHE, Xtr=Xtr_all, Xte=Xte_all)

XV_rich, XU_rich, XT_rich = Xtr_all[verified_mask], Xtr_all[~verified_mask], Xte_all

def extract_patches(path):
    try:
        audio, rate = sf.read(str(path), dtype="float32")
        if audio.ndim > 1: audio = audio.mean(axis=1)
        mel = librosa.feature.melspectrogram(y=audio, sr=rate, n_fft=1024, hop_length=max(1, round(rate * 0.012)), n_mels=40)
        logmel = librosa.power_to_db(mel, ref=np.max).T
        views = []
        for width in (8, 32):
            hop = width // 2
            frames = np.pad(logmel, ((0, max(0, width - len(logmel))), (0, 0)), mode="edge")
            starts = list(range(0, len(frames) - width + 1, hop))
            if starts[-1] != len(frames) - width: starts.append(len(frames) - width)
            blocks = np.stack([frames[i:i+width] for i in starts])
            half = width // 2
            views.append(np.hstack([blocks.mean(1), blocks.std(1), blocks[:, half:].mean(1) - blocks[:, :half].mean(1)]).astype(np.float32))
        return views
    except Exception as e:
        return None

PATCH_CACHE = WORK / "p4_patches.joblib"
if PATCH_CACHE.exists():
    pdata = load(PATCH_CACHE)
    train_patches, test_patches = pdata["train_patches"], pdata["test_patches"]
else:
    train_patches = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_patches)(TRAIN_AUDIO / f"{fid}.wav") for fid in train_df["id"]
    )
    test_patches = Parallel(n_jobs=-1, batch_size=16, verbose=1)(
        delayed(extract_patches)(TEST_AUDIO / f"{fid}.wav") for fid in test_df["id"]
    )
    dump({"train_patches": train_patches, "test_patches": test_patches}, PATCH_CACHE, compress=3)

all_patches = train_patches + test_patches

## 2. Multi-Scale Vocabularies & Dual Kernels (VLAD + PCA Fisher)

In [ ]:
rng = np.random.default_rng(42)
fisher_views   = []
residual_views = []
encoded_tpp    = []

for view in range(2):
    samples = []
    for i in unchecked_ids:
        frames = train_patches[i][view]
        samples.append(frames[rng.choice(len(frames), min(20, len(frames)), replace=False)])
    pool = np.vstack(samples)
    scaler = StandardScaler().fit(pool)

    # 1. TPP Dictionary (96 clusters)
    dictionary = MiniBatchKMeans(n_clusters=96, batch_size=2048, n_init=3, random_state=42)
    dictionary.fit(scaler.transform(pool))
    centers = dictionary.cluster_centers_

    def encode_tpp_single(record):
        frames = scaler.transform(record[view])
        T = len(frames)
        def h(f):
            if len(f) == 0: return np.zeros(len(centers))
            d = cdist(f, centers, 'sqeuclidean')
            nr = np.argpartition(d, 2, axis=1)[:, :3]
            w = 1.0 / np.maximum(np.take_along_axis(d, nr, axis=1), 1e-8)
            w /= w.sum(axis=1, keepdims=True)
            hi = np.bincount(nr.ravel(), weights=w.ravel(), minlength=len(centers))
            s = hi.sum()
            return np.sqrt(hi / s) if s > 0 else hi
        t1, t2 = T // 3, 2 * T // 3
        return np.concatenate([h(frames), h(frames[:t1]), h(frames[t1:t2]), h(frames[t2:])])

    encoded_tpp.append(np.vstack([encode_tpp_single(r) for r in all_patches]))

    # 2. VLAD Residual Dictionary (32 clusters)
    residual_dict = MiniBatchKMeans(n_clusters=32, batch_size=2048, n_init=3, random_state=17)
    residual_dict.fit(scaler.transform(pool))
    centers_r = residual_dict.cluster_centers_

    def encode_residual(record):
        frames = scaler.transform(record[view]).astype(np.float32)
        assignment = cdist(frames, centers_r, 'sqeuclidean').argmin(axis=1)
        sums = np.zeros_like(centers_r)
        np.add.at(sums, assignment, frames - centers_r[assignment])
        sums /= np.maximum(np.linalg.norm(sums, axis=1, keepdims=True), 1e-8)
        vec = np.sign(sums.ravel()) * np.sqrt(np.abs(sums.ravel()))
        return vec / max(np.linalg.norm(vec), 1e-8)

    residual_views.append(np.stack([encode_residual(r) for r in all_patches]))

    # 3. PCA-Whitened GMM Fisher Vectors (24 PCA dims + 32 diagonal GMM components)
    projection = PCA(n_components=24, whiten=True, svd_solver='randomized', random_state=81)
    projected  = projection.fit_transform(scaler.transform(pool))
    mixture    = GaussianMixture(32, covariance_type='diag', reg_covar=0.01, max_iter=70, random_state=81)
    mixture.fit(projected)

    def fisher(record):
        x = projection.transform(scaler.transform(record[view]))
        q = mixture.predict_proba(x)
        mass = q.sum(axis=0)[:, None]
        moments = q.T @ x
        mu, var = mixture.means_, mixture.covariances_
        norm = len(x) * np.sqrt(mixture.weights_[:, None])
        first = (moments - mass * mu) / (norm * np.sqrt(var))
        second = ((q.T @ (x * x) - 2 * mu * moments + mass * mu**2) / var - mass) / (np.sqrt(2) * norm)
        v = np.concatenate([first.ravel(), second.ravel()])
        v = np.sign(v) * np.sqrt(np.abs(v))
        return (v / max(np.linalg.norm(v), 1e-8)).astype(np.float32)

    fisher_views.append(np.stack([fisher(r) for r in all_patches]))

acoustic_tpp = np.hstack(encoded_tpp).astype('float32')
train_a_tpp, test_a_tpp = np.split(acoustic_tpp, [len(train_df)])
av_tpp, au_tpp = train_a_tpp[verified_mask], train_a_tpp[~verified_mask]

vlad_kernel = linear_kernel(np.column_stack(residual_views), np.column_stack(residual_views))
del residual_views

fisher_kernel = linear_kernel(np.column_stack(fisher_views), np.column_stack(fisher_views))
del fisher_views

## 3. Retraining All Models on 100% of Verified Ground Truth (ALL 620 Samples)

In [ ]:
def make_rich_models():
    svm = lambda C: make_pipeline(StandardScaler(), SVC(C=C, kernel="rbf", gamma="scale", class_weight="balanced", probability=True, random_state=42))
    lr  = lambda C: make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=3000))
    rf  = lambda n: RandomForestClassifier(n_estimators=n, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=-1)
    et  = lambda n: ExtraTreesClassifier(n_estimators=n, class_weight="balanced", min_samples_leaf=2, random_state=42, n_jobs=-1)
    return [("svm3",  svm(3),  "svc__sample_weight", 0.25),
            ("svm10", svm(10), "svc__sample_weight", 0.15),
            ("lr",    lr(0.1), "logisticregression__sample_weight", 0.10),
            ("rf",    rf(500), "sample_weight", 0.25),
            ("et",    et(500), "sample_weight", 0.25)]

def fit_predict_rich_all(Xtr, ytr, wtr, X_list):
    outs = [np.zeros((len(X), K)) for X in X_list]
    for name, m, wk, a in make_rich_models():
        kw = {wk: wtr} if wtr is not None else {}
        m.fit(Xtr, ytr, **kw)
        for o, X in zip(outs, X_list): o[:, m.classes_] += a * m.predict_proba(X)
    return outs

# Stage 1: Fit on ALL 620 verified
Pu_all = fit_predict_rich_all(XV_rich, yv, None, [XU_rich])[0]
s_all  = Pu_all[np.arange(len(yu)), yu]
rank_all = (Pu_all > s_all[:, None]).sum(1)
keep_all = rank_all < 1
print(f"Agreement on full unchecked pool: {keep_all.sum()} samples kept")

# Stage 2: Fit on ALL 620 verified + kept unchecked
Xa_rich = np.vstack([XV_rich, XU_rich[keep_all]])
ya_rich = np.concatenate([yv, yu[keep_all]])
wa_rich = np.concatenate([np.ones(len(yv)), np.full(keep_all.sum(), 0.50)])
base_test = fit_predict_rich_all(Xa_rich, ya_rich, wa_rich, [XT_rich])[0]

# Stream B: TPP Patch Model on ALL 620 verified + kept unchecked
Xb_tpp = np.vstack([av_tpp, au_tpp[keep_all]])
yb_tpp = np.concatenate([yv, yu[keep_all]])
wb_tpp = np.concatenate([np.ones(len(yv)), np.full(keep_all.sum(), 0.50)])
m_patch = SVC(C=12, kernel="rbf", probability=True, random_state=42)
m_patch.fit(Xb_tpp, yb_tpp, sample_weight=wb_tpp)
local_test = m_patch.predict_proba(test_a_tpp)

# Stream C1 & C2: Dual Kernels on ALL 620 verified + unchecked
all_train_ids = np.r_[verified_ids, unchecked_ids]
all_labels    = np.r_[yv, yu]
all_weights   = np.r_[np.ones(len(yv)), np.where(keep_all, 0.50, 0.08)]

m_vlad = SVC(kernel="precomputed", C=12, probability=True, random_state=42)
m_vlad.fit(vlad_kernel[np.ix_(all_train_ids, all_train_ids)], all_labels, sample_weight=all_weights)
vlad_test = m_vlad.predict_proba(vlad_kernel[np.ix_(test_ids, all_train_ids)])

m_fisher = SVC(kernel="precomputed", C=12, probability=True, random_state=42)
m_fisher.fit(fisher_kernel[np.ix_(all_train_ids, all_train_ids)], all_labels, sample_weight=all_weights)
fisher_test = m_fisher.predict_proba(fisher_kernel[np.ix_(test_ids, all_train_ids)])

## 4. Compound Stream Blending & T=1.25 EM Prior Calibration

In [ ]:
def sharpen(P, t):
    Q = np.clip(P, 1e-9, None) ** t
    return Q / Q.sum(1, keepdims=True)

def em_prior(P, iters=200, floor=0.25):
    pi = np.full(K, 1.0 / K)
    for _ in range(iters):
        Q = P * (pi * K)[None, :]
        Q /= Q.sum(1, keepdims=True)
        pi = Q.mean(0)
    pi = np.clip(pi, floor / K, None)
    return pi / pi.sum()

def adjust(P, pi):
    Q = P * pi[None, :]
    return Q / Q.sum(1, keepdims=True)

# Compound Blend Formula
ref  = 0.65 * base_test + 0.35 * local_test
raw1 = 0.70 * ref + 0.30 * vlad_test
raw2 = 0.70 * raw1 + 0.30 * fisher_test

# Temperature sharpening (T=1.25 conservative calibration) & test-set EM prior matching
temp   = 1.25
P_cal  = sharpen(raw2, temp)
pi_hat = em_prior(P_cal)
final_probs = adjust(P_cal, pi_hat ** 0.5)

top3 = np.argsort(-final_probs, axis=1)[:, :3]
sub = pd.DataFrame({
    "id": test_df["id"],
    "label": [" ".join(classes[r]) for r in top3]
})
sub = sample_df[["id"]].merge(sub, on="id", how="left", validate="one_to_one")

assert sub.shape == (1592, 2)
assert (sub["id"] == sample_df["id"]).all()
assert sub["label"].notna().all()

sub.to_csv("submission.csv", index=False)
if WORK != Path("."):
    sub.to_csv(WORK / "submission.csv", index=False)

print("Saved submission.csv successfully (1592 predictions, Public LB 0.84353)!")
print(sub.head(10))